# IT Operations Intelligence
## Fase 11-12: Exportación de Resultados y Conclusiones

**Objetivo:** Verificar los resultados obtenidos, exportar los datos finales y presentar las conclusiones del proyecto.

In [ ]:
# --- Importación de librerías ---
import pandas as pd  # Librería para manipulación y análisis de datos estructurados (DataFrames, CSV)
import os  # Librería para interactuar con el sistema operativo (rutas de archivos, directorios)
import pickle  # Librería para serializar y deserializar objetos de Python (modelos entrenados)


In [ ]:
# --- Listar archivos generados ---
ruta_final = "../data/final"  # Define la ruta relativa al directorio donde se almacenan los archivos de salida finales

archivos = os.listdir(ruta_final)  # Obtiene una lista con los nombres de todos los archivos en el directorio data/final

print("Archivos en data/final:")  # Imprime el encabezado que introduce la lista de archivos encontrados

for archivo in archivos:  # Itera sobre cada elemento de la lista de archivos obtenida
    print(f"  - {archivo}")  # Imprime el nombre del archivo con un guion para formato de lista


### Análisis de Archivos Generados

##### Los archivos encontrados en el directorio `data/final` representan los productos finales del pipeline de Machine Learning. Cada uno cumple un rol específico: el archivo `comparacion_modelos.csv` almacena las métricas de rendimiento de todos los modelos evaluados, permitiendo seleccionar el mejor; `predicciones_incidencias.csv` contiene las predicciones del modelo final sobre los datos de prueba; y archivos adicionales como datasets procesados o características derivadas respaldan la reproducibilidad del análisis.

In [ ]:
# --- Cargar y mostrar comparación de modelos ---
comparacion = pd.read_csv("../data/final/comparacion_modelos.csv")  # Lee el CSV con las métricas de rendimiento (Accuracy, Precision, Recall, F1) de cada modelo entrenado

# Muestra el DataFrame completo para inspeccionar visualmente los resultados de todos los algoritmos
comparacion


### Análisis de Comparación de Modelos

##### El DataFrame de comparación presenta las métricas clave (Accuracy, Precision, Recall, F1-Score) para cada algoritmo entrenado. Permite identificar visualmente cuál modelo ofrece el mejor equilibrio entre las distintas métricas. Un modelo con alto Recall pero baja Precision puede generar muchos falsos positivos, mientras que uno con alta Precision pero bajo Recall dejaría de detectar incidencias críticas. La selección final debe alinearse con el costo operativo de cada tipo de error.

In [ ]:
# --- Cargar y mostrar predicciones ---
# Lee el CSV con las predicciones generadas por el modelo final sobre el conjunto de prueba
predicciones = pd.read_csv("../data/final/predicciones_incidencias.csv")  

# Muestra las primeras 5 filas del DataFrame para revisar la estructura general de las predicciones
predicciones.head()


### Análisis de Predicciones Generadas

##### El DataFrame de predicciones contiene para cada ticket de prueba la probabilidad asignada por el modelo y la clasificación final (Cumple o No Cumple SLA). Las columnas incluyen el identificador del ticket, las características utilizadas, la probabilidad estimada y la etiqueta predicha. Esta estructura permite auditar individualmente cada predicción y exportar los resultados para integrarlos con sistemas de monitoreo o plataformas de ticketing.

In [ ]:
# Abrir el archivo pickle del modelo entrenado en modo lectura binaria
with open("../models/modelo_sla.pkl", "rb") as f:

    # Deserializar el objeto del modelo desde el archivo pickle
    modelo = pickle.load(f)

# Imprimir el nombre de la clase del modelo cargado para confirmar el algoritmo
print(f"Modelo cargado: {type(modelo).__name__}")


### Verificación del Modelo Cargado

##### El modelo serializado ha sido cargado exitosamente desde el archivo pickle. La salida confirma el tipo de algoritmo utilizado (por ejemplo, RandomForestClassifier, GradientBoostingClassifier, etc.), lo cual es fundamental para documentar qué técnica se empleó en la implementación final. Este paso valida que el artefacto del modelo está intacto y listo para ser usado en predicciones sobre nuevos datos.

In [ ]:
# Mostrar dimensiones de la tabla de comparación de modelos
print("Dimensiones de comparación de modelos:", comparacion.shape)

# Mostrar dimensiones de la tabla de predicciones
print("Dimensiones de predicciones:", predicciones.shape)


### Dimensiones de los Datos Exportados

##### Las dimensiones de los DataFrames indican el volumen total de registros procesados y exportados. La comparación de modelos contiene tantas filas como algoritmos fueron evaluados durante el experimento, mientras que las predicciones contienen tantas filas como registros en el conjunto de prueba. Verificar estas dimensiones asegura que no hubo pérdida de datos durante la exportación y que los archivos contienen exactamente la información esperada.

In [ ]:
# Imprimir encabezado del conteo de predicciones por clase
print("Distribución de predicciones:")

# Contar y mostrar cuántos tickets fueron clasificados en cada categoría
print(predicciones["Prediccion"].value_counts())

# Línea en blanco para separar secciones
print()

# Imprimir encabezado de porcentajes
print("Porcentaje:")

# Calcular y mostrar el porcentaje de cada clase redondeado a 2 decimales
print(round(predicciones["Prediccion"].value_counts(normalize=True) * 100, 2))


### Distribución de Predicciones del Modelo

##### La distribución muestra cuántos tickets fueron clasificados en cada categoría (Cumple SLA vs No Cumple SLA). El porcentaje revela el desbalance natural del problema: la mayoría de los tickets suelen cumplir con el SLA, pero aquellos que no cumplen representan el mayor riesgo operativo. Un modelo que predice correctamente la clase minoritaria (No Cumple SLA) aporta el mayor valor de negocio, ya que permite tomar acciones preventivas sobre esos tickets críticos antes de que se materialice el incumplimiento.

### Fase 11: Insights de Negocio

**Hallazgo 1:** La prioridad del ticket es el factor más determinante en el cumplimiento de SLA. Los tickets de alta prioridad tienen mayor probabilidad de cumplimiento.

**Hallazgo 2:** El número de reasignaciones está fuertemente correlacionado con el incumplimiento de SLA. A más reasignaciones, mayor riesgo.

**Hallazgo 3:** El tiempo de resolución es un indicador clave. Tickets con tiempos de resolución prolongados tienen mayor probabilidad de incumplir el SLA.

**Hallazgo 4:** La combinación de impacto, urgencia y prioridad (criticality_score) ofrece una mejor capacidad predictiva que el análisis individual de cada variable.

---

### Fase 12: Conclusiones y Recomendaciones

**Conclusiones:**

1. **Desbalance estructural del problema:** El 93.5% de los tickets cumplen con el SLA, mientras que solo el 6.5% lo incumplen. Este desbalance es inherente al dominio y requiere técnicas especializadas (SMOTE, class_weight, umbrales ajustables) para evitar que los modelos ignoren la clase minoritaria, que es precisamente la de mayor interés operativo.

2. **Efectividad de los modelos ensemble:** Los modelos basados en árboles (Random Forest con 200 estimadores, Gradient Boosting con profundidad 5) superaron en rendimiento general a la Regresión Logística y la Red Neuronal MLP. Gradient Boosting alcanzó el mejor F1-Score (0.9661) y ROC-AUC (0.9056), demostrando que los métodos secuenciales que corrigen errores progresivamente se adaptan bien a la estructura de los datos de incidencias IT.

3. **Importancia de la ingeniería de atributos:** La creación de variables como resolution_time_hours, criticality_score (promedio de impacto, urgencia y prioridad) y problematic_ticket (basado en reasignaciones y reaperturas) mejoró sustancialmente la capacidad predictiva del modelo. Las tres variables más importantes según Gradient Boosting fueron sys_mod_count (53.5% de importancia relativa), resolution_time_hours (23.4%) y reassignment_count (6.9%), lo que confirma que el historial operativo del ticket es más predictivo que sus metadatos estáticos.

4. **Limitaciones en la detección de la clase minoritaria:** A pesar del balanceo con SMOTE y el uso de class_weight, los modelos mostraron recall limitado para la clase de incumplimiento (0). La Regresión Logística logró el mejor recall (75%) pero con baja precisión (21%), mientras que Gradient Boosting priorizó la exactitud global sacrificando la detección de incumplimientos. Esto sugiere que se necesitan estrategias adicionales como optimización de umbrales, cost-sensitive learning o segmentación de tickets por perfil de riesgo.

5. **Relaciones no lineales entre variables:** El análisis exploratorio reveló que variables como open_hour, open_dayofweek y open_month tienen correlaciones débiles con el SLA de forma individual, pero al combinarse con otras características en modelos ensemble adquieren poder predictivo. Esto valida el uso de modelos complejos capaces de capturar interacciones no lineales que los modelos lineales como Regresión Logística no pueden aprovechar.

6. **Factores críticos de incumplimiento:** El número de modificaciones del ticket (sys_mod_count) domina la importancia predictiva con más de la mitad del peso total. Los tickets que requieren muchas modificaciones, reasignaciones o tienen largos tiempos de resolución son los candidatos más probables a incumplir el SLA. Estos factores son indicadores operativos tempranos que pueden monitorearse en tiempo real.

7. **Madurez de los datos:** El dataset original contenía 141,712 registros y 48 columnas tras el feature engineering, sin filas duplicadas y con valores nulos concentrados en columnas de resolución (tickets no resueltos). La calidad general de los datos permitió un modelado robusto, aunque la imputación de nulos con la mediana podría refinarse con técnicas más avanzadas como imputación predictiva o modelos de supervivencia para tickets no resueltos.

8. **Interpretabilidad vs rendimiento:** Existe un trade-off claro entre modelos interpretables (Regresión Logística) y modelos de caja negra (Gradient Boosting, MLP). La Regresión Logística ofrece coeficientes directamente interpretables que permiten entender el impacto de cada variable en el SLA, mientras que Gradient Boosting requiere técnicas adicionales como SHAP o LIME para explicar predicciones individuales. Para un entorno de producción, se recomienda mantener ambos enfoques: un modelo interpretable para auditoría y uno complejo para precisión.

**Recomendaciones:**

1. **Sistema de alertas tempranas multicapa:** Implementar un sistema que clasifique los tickets entrantes en niveles de riesgo (bajo, medio, alto) basado en la probabilidad predicha por el modelo. Los tickets con probabilidad superior al 70% de incumplimiento deberían activar alertas automáticas al equipo de soporte asignado, permitiendo intervenciones preventivas antes de que se materialice la violación del SLA.

2. **Reducción de reasignaciones mediante enrutamiento inteligente:** Dado que reassignment_count es una de las variables más predictivas, se recomienda implementar un sistema de enrutamiento basado en la categoría del ticket, la especialidad del agente y la carga de trabajo actual. Establecer un límite máximo de reasignaciones (por ejemplo, 3) antes de escalar automáticamente a un nivel superior de soporte, reduciendo así los tiempos de resolución.

3. **Monitoreo continuo de la deriva del modelo:** El rendimiento del modelo debe evaluarse periódicamente (semanal o mensual) sobre datos nuevos para detectar degradación por concept drift. Implementar un pipeline de reentrenamiento automático que actualice el modelo cuando métricas clave (F1-Score, ROC-AUC) caigan por debajo de umbrales definidos, asegurando que el sistema se adapte a cambios en los patrones de incidencias.

4. **Optimización del umbral de decisión:** Ajustar el umbral de clasificación por defecto de 0.5 a un valor más bajo (por ejemplo, 0.3 o 0.4) para mejorar el recall de la clase minoritaria, aceptando un mayor número de falsos positivos a cambio de detectar más incumplimientos reales. El umbral óptimo debe determinarse mediante análisis de costo-beneficio, ponderando el costo operativo de una falsa alarma frente al costo de un incumplimiento no detectado.

5. **Segmentación y modelos especializados por categoría:** Dado que ciertas categorías de incidentes tienen tasas de incumplimiento del 100% (aunque con bajo volumen), se recomienda entrenar modelos especializados por grupo de categorías o utilizar técnicas de meta-aprendizaje para mejorar la predicción en categorías minoritarias. Esto permitiría capturar patrones específicos que el modelo global podría estar diluyendo.

6. **Dashboard de monitorización operativa:** Crear un dashboard en tiempo real que visualice: (a) distribución de probabilidades de incumplimiento, (b) top 10 tickets en riesgo, (c) tendencias temporales de incumplimiento por hora/día/mes, (d) mapa de calor de categorías con mayor tasa de incumplimiento, y (e) evolución de las métricas del modelo. Este dashboard permitiría a los equipos de soporte tomar decisiones informadas basadas en datos.

7. **Integración con el sistema de ticketing:** Incorporar la puntuación de riesgo (probability score) como un campo adicional en el sistema de ticketing (ServiceNow, Jira, etc.), visible tanto para agentes como para supervisores. Esto permitiría priorizar automáticamente la cola de trabajo según el riesgo predicho, asignar los tickets más críticos a los agentes más experimentados y generar informes de cumplimiento predictivo.

8. **Análisis de causas raíz mediante SHAP:** Implementar explicaciones SHAP (SHapley Additive exPlanations) para cada predicción individual, permitiendo a los agentes entender por qué un ticket específico fue marcado como de alto riesgo. Esto no solo aumenta la confianza en el modelo, sino que también proporciona insights accionables: si sys_mod_count es el factor dominante, el agente sabe que debe priorizar la revisión de tickets con muchas modificaciones.

9. **Ampliación del feature engineering futuro:** Incorporar características adicionales como: (a) texto de la descripción del ticket mediante NLP para extraer temas y urgencia semántica, (b) métricas de carga de trabajo del equipo asignado, (c) estacionalidad de incidentes similares en el pasado, (d) indicadores de cumplimiento de SLA del agente asignado, y (e) datos de cambios planificados o ventanas de mantenimiento que puedan afectar la resolución.

10. **Evaluación de impacto económico:** Realizar un análisis de retorno de inversión (ROI) que cuantifique el ahorro estimado por cada punto porcentual de mejora en la detección temprana de incumplimientos de SLA. Este análisis debe considerar multas por incumplimiento, costos de escalamiento, horas hombre dedicadas a tickets problemáticos y el impacto en la satisfacción del cliente (CSAT). Los resultados servirán para justificar la inversión en mejoras continuas del sistema predictivo.
